# Persian MMS/VITS Fine-Tuning

This notebook fine-tunes Meta's `facebook/mms-tts-fas` VITS checkpoint on a subset of the Persian ManaTTS dataset. Run the cells in order. Model downloads and training can take a long time depending on the available hardware.

> The MMS weights use the CC BY-NC 4.0 license. Fine-tuned weights are intended for non-commercial use.

## 1. Locate the repository

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'run_vits_finetuning.py').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the repository.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
print(f'Repository: {REPO_ROOT}')
print(f'Python: {sys.version.split()[0]}')

## 2. Install dependencies and build monotonic alignment

This cell installs the known-compatible Python stack into the active notebook kernel and compiles the Cython alignment extension.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements-local.txt'],
    check=True,
)
subprocess.run(
    [sys.executable, 'setup.py', 'build_ext', '--inplace'],
    cwd=REPO_ROOT / 'monotonic_align',
    check=True,
)
print('Dependencies and alignment extension are ready.')

## 3. Check the available accelerator

In [ ]:
import torch

if torch.cuda.is_available():
    device_name = f'CUDA: {torch.cuda.get_device_name(0)}'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device_name = 'MPS'
else:
    device_name = 'CPU (training will be slow)'

print(f'PyTorch: {torch.__version__}')
print(f'Device: {device_name}')

## 4. Prepare the Persian training checkpoint

The public inference checkpoint contains the VITS generator. Fine-tuning also requires the original MMS discriminator, so this step downloads and combines both parts. It is skipped when the prepared checkpoint already exists.

In [ ]:
checkpoint_dir = REPO_ROOT / 'checkpoints' / 'mms-tts-fas-train'
checkpoint_dir.parent.mkdir(parents=True, exist_ok=True)

if (checkpoint_dir / 'config.json').exists():
    print(f'Checkpoint already exists: {checkpoint_dir}')
else:
    subprocess.run(
        [
            sys.executable,
            'convert_original_discriminator_checkpoint.py',
            '--language_code',
            'fas',
            '--pytorch_dump_folder_path',
            str(checkpoint_dir),
        ],
        check=True,
    )
    print(f'Prepared checkpoint: {checkpoint_dir}')

## 5. Configure the training run

The defaults use 1,000 training samples, 25 validation samples, and 20 epochs. Reduce `TRAIN_SAMPLES` and `EPOCHS` for a quick smoke test.

In [ ]:
import json

TRAIN_SAMPLES = 1_000
EVAL_SAMPLES = 25
EPOCHS = 20
TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4

base_config_path = REPO_ROOT / 'training_config_examples' / 'finetune_mms_fa_local.json'
with base_config_path.open(encoding='utf-8') as config_file:
    config = json.load(config_file)

config['train_split_name'] = f'train[{EVAL_SAMPLES}:{EVAL_SAMPLES + TRAIN_SAMPLES}]'
config['eval_split_name'] = f'train[:{EVAL_SAMPLES}]'
config['num_train_epochs'] = EPOCHS
config['per_device_train_batch_size'] = TRAIN_BATCH_SIZE
config['gradient_accumulation_steps'] = GRADIENT_ACCUMULATION_STEPS
config['model_name_or_path'] = str(checkpoint_dir)
config['output_dir'] = str(REPO_ROOT / 'outputs' / 'persian-mms-tts')

runtime_config_path = REPO_ROOT / 'outputs' / 'notebook_training_config.json'
runtime_config_path.parent.mkdir(parents=True, exist_ok=True)
with runtime_config_path.open('w', encoding='utf-8') as config_file:
    json.dump(config, config_file, ensure_ascii=False, indent=2)

print(f'Training samples: {TRAIN_SAMPLES}')
print(f'Validation samples: {EVAL_SAMPLES}')
print(f'Epochs: {EPOCHS}')
print(f'Runtime config: {runtime_config_path}')

## 6. Fine-tune the model

This is the long-running step. Checkpoints are saved under `outputs/persian-mms-tts`. Re-running the cell resumes from the latest checkpoint when one is available.

In [ ]:
environment = os.environ.copy()
environment.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
environment.setdefault('TOKENIZERS_PARALLELISM', 'false')

subprocess.run(
    [
        sys.executable,
        '-m',
        'accelerate.commands.launch',
        '--num_processes',
        '1',
        '--mixed_precision',
        'no',
        'run_vits_finetuning.py',
        str(runtime_config_path),
    ],
    env=environment,
    check=True,
)

## 7. Generate and play a speech sample

If training has not completed, this cell falls back to the base Persian MMS model.

In [ ]:
from IPython.display import Audio, display

SAMPLE_TEXT = 'سلام، این یک نمونه از تبدیل متن فارسی به گفتار است.'
fine_tuned_model = REPO_ROOT / 'outputs' / 'persian-mms-tts'
model_source = str(fine_tuned_model) if (fine_tuned_model / 'config.json').exists() else 'facebook/mms-tts-fas'
sample_path = REPO_ROOT / 'outputs' / 'notebook_sample.wav'

subprocess.run(
    [
        sys.executable,
        'scripts/infer.py',
        '--model',
        model_source,
        '--text',
        SAMPLE_TEXT,
        '--output',
        str(sample_path),
    ],
    check=True,
)
print(f'Model: {model_source}')
display(Audio(filename=str(sample_path)))